# Evaluation of Token Classification NER Model (BIO-based Encoder)

This notebook evaluates a fine-tuned **encoder-based token classification** model for financial NER.

**Approach**: Baseline Encoder (e.g., XLM-RoBERTa) → Linear Classification Head
- BIO tagging scheme (B-Entity, I-Entity, O)
- First-subword pooling: the first subword of each word is used as the word representation

### Steps:
1. **Install Dependencies**
2. **Clone/Verify** the `NER_Token_Classification` pipeline
3. **Convert Dataset**: `ner_consolidated_vi.json` → FIRE format
4. **Configure**: Modify YAML config for Vietnamese test data
5. **Run Evaluation**: Execute evaluation script

In [ ]:
# Step 1: Install dependencies
!pip install transformers accelerate seqeval torch pyyaml tqdm datasets

In [ ]:
# Step 2: Ensure the NER_Token_Classification pipeline code is available
import os

PIPELINE_DIR = "/kaggle/working/NER_Token_Classification"

if not os.path.exists(PIPELINE_DIR):
    print("Cloning NER repository...")
    !git clone https://github.com/nam13092003/NER-financial-data.git /kaggle/working/repo_tmp
    !cp -r /kaggle/working/repo_tmp/NER_Token_Classification {PIPELINE_DIR}
    !rm -rf /kaggle/working/repo_tmp
else:
    print(f"NER_Token_Classification pipeline already available at {PIPELINE_DIR}")

In [ ]:
# Step 3: Convert ner_consolidated_vi.json to FIRE format
import os
import json

DATASET_INPUT_PATH = "ner_consolidated_vi.json"
DATASET_OUTPUT_PATH = "/kaggle/working/ner_consolidated_vi_fire.json"

# Search recursively for the input file (useful on Kaggle)
actual_input_path = DATASET_INPUT_PATH
if not os.path.exists(actual_input_path):
    if os.path.exists(os.path.join("/kaggle/input", DATASET_INPUT_PATH)):
        actual_input_path = os.path.join("/kaggle/input", DATASET_INPUT_PATH)
    else:
        found = False
        for search_dir in [".", "/kaggle/input"]:
            if not os.path.exists(search_dir): continue
            for root, _, files in os.walk(search_dir):
                if os.path.basename(DATASET_INPUT_PATH) in files:
                    actual_input_path = os.path.join(root, os.path.basename(DATASET_INPUT_PATH))
                    found = True
                    break
            if found: break


def char_to_token_span(sentence: str, start_char: int, end_char: int):
    """
    Maps a character offset range [start_char, end_char) to whitespace-based word tokens.
    """
    words = sentence.split()
    word_spans = []
    current_idx = 0
    for w in words:
        start_pos = sentence.find(w, current_idx)
        if start_pos == -1:
            start_pos = current_idx
        end_pos = start_pos + len(w)
        word_spans.append((start_pos, end_pos))
        current_idx = end_pos
    
    overlapping_words = []
    for idx, (w_start, w_end) in enumerate(word_spans):
        if max(start_char, w_start) < min(end_char, w_end):
            overlapping_words.append(idx)
    
    if not overlapping_words:
        return -1, -1
    
    return overlapping_words[0], overlapping_words[-1]


print(f"Reading raw dataset from: {actual_input_path}")
with open(actual_input_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

converted_data = []
for idx, item in enumerate(raw_data):
    sentence = item.get("sentence", "")
    tokens = sentence.split()
    
    entities = []
    for ann_idx, ann in enumerate(item.get("annotations", [])):
        start_char = ann.get("start")
        end_char = ann.get("end")
        text = ann.get("text")
        label = ann.get("label")
        
        start_tok, end_tok = char_to_token_span(sentence, start_char, end_char)
        if start_tok == -1:
            continue
        
        # FIRE format: start inclusive, end exclusive
        entities.append({
            "id": f"T{ann_idx}",
            "text": text,
            "type": label,
            "start": start_tok,
            "end": end_tok + 1
        })
    
    converted_data.append({
        "tokens": tokens,
        "entities": entities
    })

# Save the FIRE-formatted dataset
os.makedirs(os.path.dirname(DATASET_OUTPUT_PATH), exist_ok=True)
with open(DATASET_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=2)

print(f"Converted {len(converted_data)} samples to FIRE format.")
print(f"Saved to: {DATASET_OUTPUT_PATH}")

In [ ]:
# Step 4: Create custom YAML config pointing to Vietnamese test data
import yaml

CONFIG_TEMPLATE = "/kaggle/working/NER_Token_Classification/configs/default.yaml"
MY_CONFIG = "/kaggle/working/my_token_cls_config.yaml"

print(f"Loading config template from: {CONFIG_TEMPLATE}")
with open(CONFIG_TEMPLATE, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Update model to match the trained checkpoint's backbone
# (Change this if you used a different encoder for training)
cfg["model"]["name"] = "xlm-roberta-base"
cfg["model"]["max_length"] = 256

# Point test_files to our converted Vietnamese FIRE dataset
cfg["data"]["test_files"] = [
    {"path": "/kaggle/working/ner_consolidated_vi_fire.json"}
]

# Save custom config
with open(MY_CONFIG, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print(f"Config saved to: {MY_CONFIG}")

## 5. Run Evaluation

Execute the token classification evaluation script.
Set `CHECKPOINT` to the path of your trained model directory.

In [ ]:
# Step 5: Run evaluation
CONFIG = "/kaggle/working/my_token_cls_config.yaml"

# Path to the trained model checkpoint
# Change this to your actual checkpoint path on Kaggle
CHECKPOINT = "/kaggle/input/datasets/nam13092003/token-cls-ner-best-model"

!cd /kaggle/working/NER_Token_Classification && python evaluate.py \
    --config {CONFIG} \
    --checkpoint {CHECKPOINT} \
    --batch_size 32 \
    --output /kaggle/working/token_cls_predictions.jsonl \
    2>&1 | tee /kaggle/working/token_cls_eval_log.txt

In [ ]:
# Step 6 (Optional): Inspect predictions
import json

PRED_PATH = "/kaggle/working/token_cls_predictions.jsonl"

if os.path.exists(PRED_PATH):
    with open(PRED_PATH, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    print(f"Total predictions: {len(lines)}")
    print("\n--- Sample Predictions (first 3) ---")
    for line in lines[:3]:
        result = json.loads(line)
        tokens = result["tokens"]
        print(f"\nTokens: {' '.join(tokens[:20])}{'...' if len(tokens) > 20 else ''}")
        print(f"  Gold entities:      {result['gold_entities']}")
        print(f"  Predicted entities: {result['predicted_entities']}")
else:
    print(f"Predictions file not found at {PRED_PATH}")

# Load summary
SUMMARY_PATH = "/kaggle/working/token_cls_predictions_summary.json"
if os.path.exists(SUMMARY_PATH):
    with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
        summary = json.load(f)
    print("\n--- Evaluation Summary ---")
    for k, v in summary.items():
        print(f"  {k}: {v}")